# 01 Data Understanding

Phase 1 reviews the manually provided LendingClub-style raw dataset, defines the starter default target, documents schema and missingness, and creates a small modeling-ready sample for the next phase. No PD model, ECL engine, or dashboard logic is built in this notebook.

## Project Context

This project is a finance and risk analytics portfolio project focused on credit risk and IFRS 9 Expected Credit Loss. Phase 1 is limited to data understanding, schema review, target definition, initial exploratory analysis, and preparation of a small modeling-ready sample.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from src.data_prep import (
    find_raw_dataset,
    get_candidate_features,
    load_raw_data,
    map_loan_status_to_target,
    summarize_schema,
)
from src.visualization import (
    plot_default_rate_by_category,
    plot_missing_values,
    plot_numeric_distribution,
    plot_target_distribution,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Dataset Loading

The dataset is detected from `data/raw/`. Plain `.csv` and compressed `.csv.gz` files are treated as CSV-style files. If multiple CSV-style files exist, the largest file is selected for Phase 1.

In [ ]:
csv_files = sorted(
    [path for path in RAW_DATA_DIR.iterdir() if path.is_file() and (path.suffix == ".csv" or path.name.endswith(".csv.gz"))],
    key=lambda path: path.stat().st_size,
    reverse=True,
)
dataset_path = find_raw_dataset()

display(pd.DataFrame({
    "file": [path.name for path in csv_files],
    "size_mb": [round(path.stat().st_size / 1_000_000, 2) for path in csv_files],
}))

if len(csv_files) > 1:
    print(f"Multiple CSV-style files found. Selected the largest file: {dataset_path.name}")
else:
    print(f"Selected dataset file: {dataset_path.name}")

SAMPLE_SIZE = 50_000
df = load_raw_data(sample_size=SAMPLE_SIZE)
print(f"Loaded {len(df):,} rows for Phase 1 exploration from {dataset_path.name}.")

## Dataset Shape and Columns

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(pd.DataFrame({"columns": df.columns.tolist()}))
display(df.head())

## Schema Summary

In [ ]:
schema_summary = summarize_schema(df)
display(schema_summary.head(40))

## Missing Value Review

In [ ]:
missing_top20 = schema_summary.sort_values("missing_pct", ascending=False).head(20)
display(missing_top20[["column", "missing_count", "missing_pct"]])

image = plot_missing_values(df, top_n=20)
image.save(FIGURES_DIR / "missing_values_top20.png")
display(image)

## Loan Status Review

In [ ]:
if "loan_status" not in df.columns:
    raise KeyError("loan_status is required for Phase 1 target definition.")

loan_status_counts = df["loan_status"].astype("string").str.strip().value_counts(dropna=False)
display(loan_status_counts.to_frame("row_count"))

## Target Definition

The starter binary target is `default_flag`.

- `default_flag = 1`: `Charged Off`, `Default`, `Late (31-120 days)`, `Does not meet the credit policy. Status:Charged Off`
- `default_flag = 0`: `Fully Paid`, `Current`, `In Grace Period`, `Late (16-30 days)`, `Does not meet the credit policy. Status:Fully Paid`
- Unknown statuses are listed and excluded from the modeling-ready sample for now.

In [ ]:
modeling_ready, unknown_statuses = map_loan_status_to_target(df)
print(f"Rows with known target mapping: {len(modeling_ready):,}")
print(f"Rows excluded because of unknown target status: {len(df) - len(modeling_ready):,}")
print("Unknown statuses:", unknown_statuses if unknown_statuses else "None")
display(modeling_ready[["loan_status", "default_flag"]].head())

## Target Distribution

In [ ]:
target_distribution = modeling_ready["default_flag"].value_counts(normalize=False).sort_index().to_frame("row_count")
target_distribution["share"] = modeling_ready["default_flag"].value_counts(normalize=True).sort_index().round(4)
display(target_distribution)

image = plot_target_distribution(modeling_ready, target_col="default_flag")
image.save(FIGURES_DIR / "target_distribution.png")
display(image)

## Candidate Risk Drivers

The candidate feature list is intentionally simple for Phase 1 and only includes columns that exist in the selected dataset. Final model features will be decided in Phase 2.

In [ ]:
candidate_features = get_candidate_features(modeling_ready)
display(pd.DataFrame({"candidate_feature": candidate_features}))

modeling_sample_columns = ["default_flag"] + candidate_features
modeling_sample = modeling_ready[modeling_sample_columns].head(50_000).copy()
modeling_sample_path = PROCESSED_DATA_DIR / "modeling_sample.csv"
modeling_sample.to_csv(modeling_sample_path, index=False)
print(f"Saved modeling-ready sample: {modeling_sample_path}")
print(f"Modeling sample shape: {modeling_sample.shape}")

## Initial EDA

In [ ]:
category_checks = ["grade", "sub_grade", "term", "home_ownership", "purpose", "verification_status"]
saved_category_figures = []

for column in category_checks:
    if column in modeling_ready.columns:
        rate_table = (
            modeling_ready.groupby(column, dropna=False)["default_flag"]
            .agg(default_rate="mean", row_count="size")
            .sort_values("default_rate", ascending=False)
        )
        rate_table["default_rate"] = (rate_table["default_rate"] * 100).round(2)
        print(f"Default rate by {column}")
        display(rate_table.head(25))

        image = plot_default_rate_by_category(modeling_ready, column, target_col="default_flag", top_n=20)
        figure_path = FIGURES_DIR / f"default_rate_by_{column}.png"
        image.save(figure_path)
        saved_category_figures.append(figure_path.name)
        display(image)
    else:
        print(f"Skipped {column}: column not available.")

print("Saved category figures:", saved_category_figures)

In [ ]:
numeric_checks = ["loan_amnt", "int_rate", "annual_inc", "dti"]
saved_numeric_figures = []

for column in numeric_checks:
    if column in modeling_ready.columns:
        numeric_summary = pd.to_numeric(modeling_ready[column], errors="coerce").describe().to_frame(column)
        print(f"Numeric summary for {column}")
        display(numeric_summary)

        image = plot_numeric_distribution(modeling_ready, column)
        figure_path = FIGURES_DIR / f"numeric_distribution_{column}.png"
        image.save(figure_path)
        saved_numeric_figures.append(figure_path.name)
        display(image)
    else:
        print(f"Skipped {column}: column not available.")

print("Saved numeric figures:", saved_numeric_figures)

## Initial Business Observations

In [ ]:
observations = []
default_rate = modeling_ready["default_flag"].mean()
observations.append(f"The Phase 1 sample default rate is {default_rate:.2%} across {len(modeling_ready):,} rows with mapped target labels.")

for column in ["grade", "sub_grade", "term", "home_ownership", "purpose", "verification_status"]:
    if column in modeling_ready.columns:
        grouped = modeling_ready.groupby(column, dropna=False)["default_flag"].agg(default_rate="mean", row_count="size")
        grouped = grouped[grouped["row_count"] >= 100]
        if not grouped.empty:
            high_risk = grouped.sort_values("default_rate", ascending=False).head(1)
            low_risk = grouped.sort_values("default_rate", ascending=True).head(1)
            observations.append(
                f"For {column}, the highest observed default rate among groups with at least 100 rows is "
                f"{high_risk.index[0]} at {high_risk['default_rate'].iloc[0]:.2%}; "
                f"the lowest is {low_risk.index[0]} at {low_risk['default_rate'].iloc[0]:.2%}."
            )

for observation in observations:
    print(f"- {observation}")

## Phase 1 Limitations

- This notebook uses a maximum 50,000-row sample for GitHub-friendly exploration and output creation.
- Target mapping is based on initial LendingClub-style loan status categories and should be reviewed before modeling.
- No PD model has been trained.
- No LGD, EAD, staging, macroeconomic overlays, or IFRS 9 ECL assumptions have been implemented.
- Some columns may have high missingness and need feature-level treatment in Phase 2.

## Next Steps for Phase 2 PD Modeling

- Confirm the target definition and treatment of current or delinquent-but-not-defaulted accounts.
- Split the modeling sample into train and validation sets.
- Build preprocessing for categorical and numeric variables.
- Train a simple benchmark PD model.
- Evaluate discrimination, calibration, and business interpretability before ECL work begins.